# 第1回：予測モデルを動かしてみる

**今日の問い：予測モデルは、データを受け取って何を返しているのか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

        - 特徴量・目的変数・学習・予測を、画面上の入出力と結びつける
- クラス予測と確率予測を区別する
- 設定を1つ変え、検証結果の変化を言葉にする

        ### 進み方

        `CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
        `SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

        ### 先に押さえる言葉

        - 特徴量：予測時点でモデルへ渡す情報
- 目的変数：予測したい答え
- 学習：既知データから関係を推定する処理
- 推論：学習済みモデルを未知データへ使う処理

        > **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## まず完成済みモデルを動かす

`X`はモデルへ渡す特徴量、`y`は答えとなる目的変数です。最初は細部を暗記せず、`fit`と`predict`の前後で何が入出力されるかを見ます。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

features = ["molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds"]
X = df[features].fillna(df[features].median())
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)
prediction = model.predict(X_valid)
print("検証データの正解率:", round(accuracy_score(y_valid, prediction), 3))
pd.DataFrame({"実際": y_valid.head(8), "予測": prediction[:8]})


## TRY

`X_valid.iloc[[0]]`をモデルへ渡し、1試料の予測を見ます。予測`0`は非活性、`1`は活性を表します。


In [ ]:
one_sample = X_valid.iloc[[0]]
print("入力した特徴量")
display(one_sample)
print("予測クラス:", model.predict(one_sample)[0])
print("活性である確率:", round(model.predict_proba(one_sample)[0, 1], 3))


## CHANGE

`max_depth=4`を`2`または`8`へ変え、正解率がどう変わるか試します。値を変えた理由と結果を1行で残してください。

## ASK COPILOT

`fit`と`predict_proba`の違いを、測定装置の校正と未知試料の測定にたとえて説明してもらいます。

## まとめ

- 特徴量はモデルへ渡す情報
- 目的変数は予測したい答え
- `fit`で関係を学び、`predict`で未知データを予測する


## DEEP DIVE：結果を一段深く読む

        次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

        ### 出力を見る観点

        - 正解率は未知データ役の検証データで確認する
- 確率0.8は『必ず活性』ではなく、モデルの確信度として扱う
- 設定変更の効果は同じ分割で比べる


In [ ]:
probability = model.predict_proba(X_valid)[:, 1]
summary = pd.DataFrame({"実際": y_valid.to_numpy(), "活性確率": probability})
display(summary.groupby("実際")["活性確率"].describe().round(3))
importance = pd.DataFrame({"特徴量": features, "重要度": model.feature_importances_})
display(importance.sort_values("重要度", ascending=False).round(3))


## よくある誤り

        - 学習データの成績を実力だと思う
- 1試料の予測だけでモデル全体を判断する
- 良い数値が出るまで設定を無計画に変える

        ## SELF-STUDY（任意・30〜60分）

        - 任意の3試料について特徴量・予測クラス・確率を1表にする
- 木の深さ2・4・8を比較し、どれを選ぶか2文で書く

        成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

        ## 振り返りチェック

        1. Xとyはそれぞれ何か
2. fitとpredictは何をするか
3. 検証データが必要なのはなぜか

        答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
